In [ ]:
import re
from collections import Counter

from datasets import load_dataset

print("Loading WikiText-2 dataset...")

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

text = "\n".join(dataset["train"]["text"])

print("Dataset Loaded Successfully!")


def preprocess_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


clean_text = preprocess_text(text)

tokens = clean_text.split()

print("Total Tokens:", len(tokens))

tokens = tokens[:100000]

print(
    "Tokens used for training:",
    len(tokens)
)


unigram_counts = Counter(tokens)
bigram_counts = Counter()
trigram_counts = Counter()


for i in range(len(tokens) - 1):

    bigram = (
        tokens[i],
        tokens[i + 1]
    )

    bigram_counts[bigram] += 1


for i in range(len(tokens) - 2):

    trigram = (
        tokens[i],
        tokens[i + 1],
        tokens[i + 2]
    )

    trigram_counts[trigram] += 1


vocabulary = set(tokens)

print(
    "Vocabulary Size:",
    len(vocabulary)
)


def predict_next_word(
    sentence,
    top_n=5
):

    sentence = preprocess_text(sentence)

    words = sentence.split()

    if len(words) == 0:
        return []


    candidates = {}


    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        total = bigram_counts[
            (word1, word2)
        ]

        if total > 0:

            for word in vocabulary:

                count = trigram_counts[
                    (word1, word2, word)
                ]

                if count > 0:

                    probability = (
                        count / total
                    )

                    candidates[word] = probability


    if len(candidates) == 0:

        previous_word = words[-1]

        total = unigram_counts[
            previous_word
        ]

        if total > 0:

            for word in vocabulary:

                count = bigram_counts[
                    (previous_word, word)
                ]

                if count > 0:

                    probability = (
                        count / total
                    )

                    candidates[word] = probability


    if len(candidates) == 0:

        total = sum(
            unigram_counts.values()
        )

        for word, count in unigram_counts.items():

            probability = (
                count / total
            )

            candidates[word] = probability


    result = sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return result[:top_n]


print("\n" + "=" * 50)
print("SMART NEXT-WORD PREDICTOR")
print("=" * 50)

print("\nType 'exit' to stop.")


while True:

    sentence = input(
        "\nEnter the sentence: "
    )

    if sentence.lower() == "exit":

        print("\nProgram stopped.")

        break


    predictions = predict_next_word(
        sentence,
        top_n=5
    )


    if not predictions:

        print(
            "No prediction available."
        )

        continue


    print(
        "\nTop Next-Word Predictions:"
    )


    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{i}. {word} - "
            f"{probability:.4f}"
        )

Loading WikiText-2 dataset...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

C:\Users\10ear\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\10ear\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset Loaded Successfully!
Total Tokens: 1753826
Tokens used for training: 100000
Vocabulary Size: 12605

SMART NEXT-WORD PREDICTOR

Type 'exit' to stop.



Enter the sentence:  machine learning



Top Next-Word Predictions:
1. and - 0.2500
2. in - 0.2500
3. the - 0.2500
4. it - 0.2500
